# Experiment: TFM Preprocesado Completo de Viñedos

Objetivo: construir un dataset limpio, trazable y listo para modelado a partir de los Excel de parcelas, campañas y geometrías.

Salidas del notebook:

- `data/processed/vinedos_modelo_limpio.csv`
- `data/processed/vinedos_modelo_diccionario.csv`
- `data/processed/vinedos_qc_resumen.md`


## 1. Setup y configuración

Este notebook está pensado para ejecutarse top-to-bottom con `uv` y Python 3.12.


In [1]:
from __future__ import annotations

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)


In [2]:
ROOT = Path('/Users/ivanr/Documents/TFM/code')
DATA_DIR = ROOT / 'data'
OUT_DIR = DATA_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'innovi': {
        'book': DATA_DIR / 'Viñedos_Innovi_corrected (1).xlsx',
        'sigpac': DATA_DIR / 'Viñedos_Innovi_corrected_sigpac (1).xlsx',
    },
    'tactic': {
        'book': DATA_DIR / 'Tactic_plantilla_viñedo_CWP_corrected (1).xlsx',
        'sigpac': DATA_DIR / 'Tactic_plantilla_viñedo_CWP_corrected_sigpac (1).xlsx',
    }
}

for k, v in FILES.items():
    print(k, 'book_exists=', v['book'].exists(), 'sigpac_exists=', v['sigpac'].exists())


innovi book_exists= True sigpac_exists= True
tactic book_exists= True sigpac_exists= True


## 2. Funciones auxiliares


In [5]:
def normalize_col(col: str) -> str:
    col = str(col).strip().lower()
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('ascii')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return re.sub(r'_+', '_', col).strip('_')


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalize_col(c) for c in out.columns]
    return out


def split_ids(value: object) -> list[str]:
    if pd.isna(value):
        return []
    txt = str(value).strip()
    if not txt:
        return []
    return [x.strip() for x in txt.split(',') if x.strip()]


def normalize_unit(value: object) -> str:
    if pd.isna(value):
        return pd.NA
    
    # Strip all Unicode whitespace and collapse, then lowercase  
    t = re.sub(r'[\s\u00a0\u3000]+', '', str(value)).lower()
    
    # Strip trailing punctuation artifacts (., ,, ), (, etc.)
    t = t.strip('.,;:()[]{}')
    
    # Canonical mapping — just the meaningful variants
    mapping = {
        'kg/ha': 'kg/ha',
        'kgha':  'kg/ha',   # missing slash
        'l/ha':  'l/ha',
        'u/ha':  'u/ha',
    }
    
    return mapping.get(t, t)


def mode_or_na(series: pd.Series):
    s = series.dropna().astype(str).str.strip()
    s = s[s != '']
    if s.empty:
        return pd.NA
    return s.mode().iloc[0]


## 3. Carga de hojas


In [6]:
def load_bundle(bundle_name: str) -> dict[str, pd.DataFrame]:
    conf = FILES[bundle_name]
    book = conf['book']
    sig = conf['sigpac']

    xls_book = pd.ExcelFile(book)
    xls_sig = pd.ExcelFile(sig)

    data = {f'book::{s}': clean_columns(pd.read_excel(book, sheet_name=s)) for s in xls_book.sheet_names}
    data.update({f'sig::{s}': clean_columns(pd.read_excel(sig, sheet_name=s)) for s in xls_sig.sheet_names})
    return data

innovi = load_bundle('innovi')
tactic = load_bundle('tactic')

print('Hojas INNOVI:')
for k, v in innovi.items():
    print(f'- {k}: {v.shape}')


Hojas INNOVI:
- book::ID_Lugar: (8341, 13)
- book::Finca viñedo: (5262, 11)
- book::Producción: (21388, 15)
- book::Fenologia: (772, 1)
- book::Fitosanitarios: (10742, 8)
- book::Fertilizantes: (1978, 14)
- book::Series temporales: (27, 7)
- sig::parcelas_sigpac: (3498, 7)
- sig::recintos_sigpac: (7651, 8)


## 4. Preprocesado base (ID_Lugar, Producción y SIGPAC)


In [7]:
id_lugar = innovi['book::ID_Lugar'] if 'book::ID_Lugar' in innovi else innovi['book::id_lugar']
produccion = innovi['book::Producción'] if 'book::Producción' in innovi else innovi['book::produccion']
finca = innovi['book::Finca viñedo'] if 'book::Finca viñedo' in innovi else innovi['book::finca_vinedo']
fito = innovi['book::Fitosanitarios'] if 'book::Fitosanitarios' in innovi else innovi['book::fitosanitarios']
ferti = innovi['book::Fertilizantes'] if 'book::Fertilizantes' in innovi else innovi['book::fertilizantes']

recintos_sig = innovi['sig::recintos_sigpac']
parcelas_sig = innovi['sig::parcelas_sigpac']

# Renombre a campos estándar
rename_id = {
    'id_lugar': 'id_lugar',
    'superficie_ha': 'superficie_ha',
    'provincia': 'provincia',
    'municipio': 'municipio',
    'agregado': 'agregado',
    'zona': 'zona',
    'poligono': 'poligono',
    'parcela': 'parcela',
    'recinto': 'recinto',
}
id_lugar = id_lugar.rename(columns=rename_id)

rename_prod = {
    'id_lugar': 'id_lugar_raw',
    'fecha_inicio_cosecha': 'fecha_inicio_cosecha',
    'fecha_fin_cosecha': 'fecha_fin_cosecha',
    'produccion_kg': 'produccion_kg',
    'grados_brix': 'grados_brix',
    'grado_alcoholico': 'grado_alcoholico',
    'destino_del_cultivo': 'destino_cultivo',
}
produccion = produccion.rename(columns=rename_prod)

for c in ['produccion_kg', 'grados_brix', 'grado_alcoholico']:
    if c in produccion.columns:
        produccion[c] = pd.to_numeric(produccion[c], errors='coerce')

produccion['fecha_inicio_cosecha'] = pd.to_datetime(produccion['fecha_inicio_cosecha'], errors='coerce')
produccion['fecha_fin_cosecha'] = pd.to_datetime(produccion['fecha_fin_cosecha'], errors='coerce')
produccion['campania'] = produccion['fecha_inicio_cosecha'].dt.year.astype('Int64')

print('id_lugar shape:', id_lugar.shape)
print('produccion shape:', produccion.shape)


id_lugar shape: (8341, 13)
produccion shape: (21388, 16)


## 5. Enlace con geometrías SIGPAC y calidad de cobertura


In [9]:
key_parcela = ['provincia', 'municipio', 'agregado', 'zona', 'poligono', 'parcela']
key_recinto = key_parcela + ['recinto']

id_geo = id_lugar.copy()

rec_geom = recintos_sig[key_recinto + ['geometry']].drop_duplicates().rename(columns={'geometry': 'geometry_recinto'})
par_geom = parcelas_sig[key_parcela + ['geometry']].drop_duplicates().rename(columns={'geometry': 'geometry_parcela'})

id_geo = id_geo.merge(rec_geom, on=key_recinto, how='left')
id_geo = id_geo.merge(par_geom, on=key_parcela, how='left')
id_geo['geometry_wkt'] = id_geo['geometry_recinto'].combine_first(id_geo['geometry_parcela'])
id_geo['has_geometry'] = id_geo['geometry_wkt'].notna()

print('Cobertura geometría por ID_lugar:', round(id_geo['has_geometry'].mean() * 100, 2), '%')


Cobertura geometría por ID_lugar: 99.51 %


## 6. Resolver `ID_lugar` multi-valor en Producción


In [10]:
prod = produccion.copy().reset_index(drop=True)
prod['prod_row_id'] = np.arange(len(prod), dtype=np.int64)
prod['id_list'] = prod['id_lugar_raw'].apply(split_ids)
prod['n_ids_en_fila'] = prod['id_list'].str.len()
prod['es_multi_id'] = prod['n_ids_en_fila'] > 1

exploded = prod[['prod_row_id', 'id_list']].explode('id_list').rename(columns={'id_list': 'id_lugar'})
exploded['id_lugar'] = exploded['id_lugar'].astype(str).str.strip()

id_surface = id_geo[['id_lugar', 'superficie_ha', 'has_geometry']].drop_duplicates('id_lugar')
exploded = exploded.merge(id_surface, on='id_lugar', how='left')

# Superficie total por fila de producción
surf_by_row = exploded.groupby('prod_row_id', as_index=False).agg(
    superficie_total_ha=('superficie_ha', 'sum'),
    n_ids_match_superficie=('superficie_ha', lambda s: s.notna().sum()),
    n_ids_match_geometria=('has_geometry', lambda s: s.fillna(False).sum()),
)

prod = prod.merge(surf_by_row, on='prod_row_id', how='left')
prod['ratio_ids_con_superficie'] = prod['n_ids_match_superficie'] / prod['n_ids_en_fila'].replace(0, np.nan)
prod['ratio_ids_con_geometria'] = prod['n_ids_match_geometria'] / prod['n_ids_en_fila'].replace(0, np.nan)

print('Filas producción:', len(prod))
print('Filas multi-id:', int(prod['es_multi_id'].sum()))
print('Filas con superficie calculable:', int(prod['superficie_total_ha'].notna().sum()))


Filas producción: 21388
Filas multi-id: 8149
Filas con superficie calculable: 21388


## 7. Features de finca (agregadas por fila de producción)


In [11]:
finca_std = finca.copy()
if 'id_lugar' not in finca_std.columns:
    # seguridad por si cambia nomenclatura
    alt = [c for c in finca_std.columns if 'id_lugar' in c]
    if alt:
        finca_std = finca_std.rename(columns={alt[0]: 'id_lugar'})

finca_keep = ['id_lugar', 'tipo_de_certificacion_manejo', 'variedad_comercial_principal', 'secano']
finca_keep = [c for c in finca_keep if c in finca_std.columns]
finca_std = finca_std[finca_keep].drop_duplicates()

exp_finca = exploded[['prod_row_id', 'id_lugar']].merge(finca_std, on='id_lugar', how='left')

agg_finca = exp_finca.groupby('prod_row_id', as_index=False).agg(
    certificacion_modo=('tipo_de_certificacion_manejo', mode_or_na),
    variedad_modo=('variedad_comercial_principal', mode_or_na),
    secano_modo=('secano', mode_or_na),
    n_variedades_distintas=('variedad_comercial_principal', lambda s: s.dropna().astype(str).str.strip().replace('', np.nan).dropna().nunique()),
)

prod = prod.merge(agg_finca, on='prod_row_id', how='left')
prod[['n_variedades_distintas']] = prod[['n_variedades_distintas']].fillna(0)


## 8. Features de manejo anual (fitosanitarios + fertilización)


In [ ]:
fito_std = fito.copy()
ferti_std = ferti.copy()

# Normalización mínima de unidades
if 'unidad_dosis' in fito_std.columns:
    fito_std['unidad_dosis_norm'] = fito_std['unidad_dosis'].apply(normalize_unit)
if 'unidad_dosis' in ferti_std.columns:
    ferti_std['unidad_dosis_norm'] = ferti_std['unidad_dosis'].apply(normalize_unit)

# Año para agregación
def add_year(df: pd.DataFrame, date_col: str, out_col: str = 'campania') -> pd.DataFrame:
    out = df.copy()
    out[out_col] = pd.to_datetime(out[date_col], errors='coerce').dt.year.astype('Int64')
    return out

fito_std = add_year(fito_std, 'fecha_inicio', 'campania')
ferti_std = add_year(ferti_std, 'fecha_inicio', 'campania')

if 'dosis_fitosanitario' in fito_std.columns:
    fito_std['dosis_fitosanitario'] = pd.to_numeric(fito_std['dosis_fitosanitario'], errors='coerce')
if 'dosis_fertilizante' in ferti_std.columns:
    ferti_std['dosis_fertilizante'] = pd.to_numeric(ferti_std['dosis_fertilizante'], errors='coerce')

fito_agg = fito_std.groupby(['id_lugar', 'campania'], as_index=False).agg(
    n_tratamientos_fito=('id_lugar', 'size'),
    dosis_fito_total=('dosis_fitosanitario', 'sum') if 'dosis_fitosanitario' in fito_std.columns else ('id_lugar', 'size'),
)

ferti_agg = ferti_std.groupby(['id_lugar', 'campania'], as_index=False).agg(
    n_aplicaciones_ferti=('id_lugar', 'size'),
    dosis_ferti_total=('dosis_fertilizante', 'sum') if 'dosis_fertilizante' in ferti_std.columns else ('id_lugar', 'size'),
)

exp_year = exploded.merge(prod[['prod_row_id', 'campania']], on='prod_row_id', how='left')

exp_year = exp_year.merge(fito_agg, on=['id_lugar', 'campania'], how='left')
exp_year = exp_year.merge(ferti_agg, on=['id_lugar', 'campania'], how='left')

for c in ['n_tratamientos_fito', 'dosis_fito_total', 'n_aplicaciones_ferti', 'dosis_ferti_total']:
    if c in exp_year.columns:
        exp_year[c] = exp_year[c].fillna(0)

manejo_by_row = exp_year.groupby('prod_row_id', as_index=False).agg(
    n_tratamientos_fito=('n_tratamientos_fito', 'sum'),
    dosis_fito_total=('dosis_fito_total', 'sum'),
    n_aplicaciones_ferti=('n_aplicaciones_ferti', 'sum'),
    dosis_ferti_total=('dosis_ferti_total', 'sum'),
)

prod = prod.merge(manejo_by_row, on='prod_row_id', how='left')


## 9. Targets y dataset final


In [ ]:
# Limpieza de grado: 0 o negativos se consideran no válidos para calidad
prod['grado_alcoholico_limpio'] = prod['grado_alcoholico'].where(prod['grado_alcoholico'] > 0, np.nan)

prod['yield_kg_ha'] = prod['produccion_kg'] / prod['superficie_total_ha']
prod['kgdegree'] = prod['produccion_kg'] * prod['grado_alcoholico_limpio']
prod['kgdegree_ha'] = prod['kgdegree'] / prod['superficie_total_ha']

# Selección de columnas para modelado
model_df = prod[[
    'prod_row_id',
    'id_lugar_raw',
    'n_ids_en_fila',
    'es_multi_id',
    'campania',
    'fecha_inicio_cosecha',
    'fecha_fin_cosecha',
    'destino_cultivo',
    'produccion_kg',
    'grados_brix',
    'grado_alcoholico',
    'grado_alcoholico_limpio',
    'superficie_total_ha',
    'ratio_ids_con_superficie',
    'ratio_ids_con_geometria',
    'certificacion_modo',
    'variedad_modo',
    'secano_modo',
    'n_variedades_distintas',
    'n_tratamientos_fito',
    'dosis_fito_total',
    'n_aplicaciones_ferti',
    'dosis_ferti_total',
    'yield_kg_ha',
    'kgdegree',
    'kgdegree_ha',
]].copy()

# Filtro mínimo de calidad para entrenamiento
train_df = model_df[
    model_df['superficie_total_ha'].notna() &
    model_df['superficie_total_ha'].gt(0) &
    model_df['produccion_kg'].notna() &
    model_df['campania'].notna()
].copy()

print('model_df rows:', len(model_df))
print('train_df rows:', len(train_df))
print('targets disponibles -> yield:', int(train_df['yield_kg_ha'].notna().sum()), 'kgdegree_ha:', int(train_df['kgdegree_ha'].notna().sum()))


## 10. Exportación de artefactos


In [ ]:
out_csv = OUT_DIR / 'vinedos_modelo_limpio.csv'
out_dict = OUT_DIR / 'vinedos_modelo_diccionario.csv'
out_qc = OUT_DIR / 'vinedos_qc_resumen.md'

train_df.to_csv(out_csv, index=False)

# Diccionario simple de variables
var_dict = pd.DataFrame({
    'columna': train_df.columns,
    'dtype': [str(train_df[c].dtype) for c in train_df.columns],
    'descripcion': [
        'ID interno de fila de produccion',
        'ID_lugar original (puede contener varios IDs separados por coma)',
        'Numero de IDs referenciados por la fila',
        'Indicador de si la fila referencia varios IDs',
        'Campania (anio de inicio de cosecha)',
        'Fecha inicio de cosecha',
        'Fecha fin de cosecha',
        'Destino del cultivo',
        'Produccion de la fila (kg)',
        'Grados brix reportados (actualmente nulos en origen)',
        'Grado alcoholico original',
        'Grado alcoholico tras limpieza (>0)',
        'Superficie total agregada de los IDs de la fila (ha)',
        'Proporcion de IDs de la fila con superficie encontrada',
        'Proporcion de IDs de la fila con geometria disponible',
        'Certificacion/modalidad mas frecuente entre IDs de la fila',
        'Variedad principal mas frecuente entre IDs de la fila',
        'Secano/regadio mas frecuente entre IDs de la fila',
        'Numero de variedades distintas entre IDs de la fila',
        'Numero total de tratamientos fitosanitarios del anio en los IDs de la fila',
        'Suma de dosis fitosanitaria del anio en los IDs de la fila',
        'Numero total de aplicaciones de fertilizacion del anio en los IDs de la fila',
        'Suma de dosis de fertilizacion del anio en los IDs de la fila',
        'Produccion por hectarea',
        'Produccion por grado (kg x grado alcoholico limpio)',
        'Kgdegree por hectarea',
    ]
})
var_dict.to_csv(out_dict, index=False)

summary_lines = [
    '# Resumen QC preprocesado',
    '',
    f'- Filas totales en produccion: {len(model_df)}',
    f'- Filas para modelado (filtro minimo): {len(train_df)}',
    f"- Filas multi-ID: {int(model_df['es_multi_id'].sum())}",
    f"- Cobertura media de superficie por fila: {model_df['ratio_ids_con_superficie'].mean():.4f}",
    f"- Cobertura media de geometria por fila: {model_df['ratio_ids_con_geometria'].mean():.4f}",
    f"- Target yield_kg_ha disponible: {int(train_df['yield_kg_ha'].notna().sum())}",
    f"- Target kgdegree_ha disponible: {int(train_df['kgdegree_ha'].notna().sum())}",
    '',
    '## Nota',
    '- `grados_brix` llega vacio en origen, por eso se usa `grado_alcoholico_limpio` para kgdegree.',
]
out_qc.write_text('
'.join(summary_lines), encoding='utf-8')

print('Exportado:', out_csv)
print('Exportado:', out_dict)
print('Exportado:', out_qc)


## 11. Validación rápida


In [ ]:
train_df.sample(5, random_state=42)
